**Welcome to the Mini Challenge!**

In this notebook, your task is to explore and report on the decision-making process of a simple CNN model trained on an image classification task. The model, trained on a varient of the MNIST dataset (a 10-class classification problem), will be loaded below along with 10 example images.

Your goal is to apply various Explainable AI (XAI) techniques to understand how the model makes decisions. Keep in mind that some XAI methods are data-agnostic. Just because you learned them in a different context doesn't mean they can't be applied to image data.

For details on grading, please refer to the "Proof of Performance" section in the EAI space.

In [ ]:
import torch
import numpy as np
import torch.nn as nn
import matplotlib.pyplot as plt
import warnings # to suppress warnings
warnings.filterwarnings("ignore") # ignore warnings

In [ ]:
# Define the CNN model
class SimpleCNN(nn.Module):
    def __init__(self):
        super(SimpleCNN, self).__init__()
        self.conv1 = nn.Conv2d(1, 32, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)
        self.fc1 = nn.Linear(64 * 7 * 7, 128)
        self.fc2 = nn.Linear(128, 10)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = x.view(-1, 64 * 7 * 7)
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x
    
# Load the models best weights
model = SimpleCNN()
model.load_state_dict(torch.load('../models/challenge_model.pth'))
model.eval()

In [ ]:
# Load data and labels
challenge_images = np.load('../data/challenge_images.npy')
challenge_labels = np.load('../data/challenge_labels.npy')
print(f"Loaded challenge images: {challenge_images.shape}, labels: {challenge_labels.shape}")

# EAI Mini Challenge

This MC applies three XAI methods (Grad-CAM, Integrated Gradients and Occlusion) to find out, what a simple CNN that was trained on a variant of the MNIST dataset has actually learned.

## Data Overview

In [ ]:
import torch.nn.functional as F

# Convert images to tensor
images_tensor = torch.tensor(challenge_images, dtype=torch.float32)

# Compute predictions
with torch.no_grad():
    outputs = model(images_tensor)
    predictions = torch.argmax(outputs, dim=1).numpy()

# Visualize images with labels and predictions
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for i, ax in enumerate(axes.flat):
    ax.imshow(challenge_images[i, 0], cmap='gray')
    ax.set_title(f"Label: {challenge_labels[i]}, Pred: {predictions[i]}")
    ax.axis('off')
plt.tight_layout()
plt.show()

The model predicts all 10 images correctly according to their labels. However, the labels do not match the digits shown in the images. For example, an image showing a 0 carries the label 1 and an image showing a 5 is labeled 3. This suggests the model may not have learned to recognize the digit shapes themselves, but rather some other signal in the data that correlates with the (incorrect) labels.

## Grad-CAM

Grad-CAM (Gradient weighted Class Activation Mapping) highlights the regions of an image that most influenced the model's prediction. It does this by computing the gradients of the predicted class score with respect to the last convolutional layer's feature maps. Regions with strong positive gradients are here considered most relevant for the decision.

In [ ]:
from captum.attr import LayerGradCam, LayerAttribution
from scipy.ndimage import gaussian_filter

# Initialize Grad-CAM on last conv layer
gradcam = LayerGradCam(model, model.conv2)

def compute_gradcam(img_np, target, smooth=False):
    img = torch.tensor(img_np, dtype=torch.float32)
    attr = gradcam.attribute(img, target=target)
    # Upsample to original image size
    heatmap = LayerAttribution.interpolate(attr, (28, 28))[0, 0].detach().numpy()
    # Keep only positive attributions and normalize
    heatmap = np.maximum(heatmap, 0)
    if heatmap.max() > 0:
        heatmap /= heatmap.max()
    if smooth:
        heatmap = gaussian_filter(heatmap, sigma=1)
    return heatmap

def plot_gradcam(smooth=False):
    fig, axes = plt.subplots(2, 5, figsize=(12, 5))
    for i, ax in enumerate(axes.flat):
        heatmap = compute_gradcam(challenge_images[i:i+1], int(predictions[i]), smooth=smooth)
        ax.imshow(challenge_images[i, 0], cmap='gray')
        ax.imshow(heatmap, cmap='jet', alpha=0.4)
        ax.set_title(f"Label: {challenge_labels[i]}, Pred: {predictions[i]}")
        ax.axis('off')
    plt.suptitle("Grad-CAM (smoothed)" if smooth else "Grad-CAM")
    plt.tight_layout()
    plt.show()

plot_gradcam(smooth=False)

Grad-CAM reveals that the model's attention is not consistently focused on the digit itself. For several images, notable activations appear in the corners, suggesting the model may be picking up on features outside the digit region. The pattern varies across images, which makes it hard to make confident conclusions at this point.

Important to state is that Grad-CAM only reflects what the last convolutional layer responds to, not the full model.

The same attributions are shown below with a Gaussian smoothing applied to the heatmap for cleaner visualization.

In [ ]:
plot_gradcam(smooth=True)

The smoothed version makes the spatial structure a bit clearer and shows that the corner activations seem to be more consistent than they first appeared.

## Integrated Gradients

Integrated Gradients attributes the model's prediction to each input pixel by computing the average gradient along a straight path from a baseline (a black image) to the actual input. Unlike Grad-CAM, it operates directly in pixelspace and does not depend on a specific layer, which makes it a more detailed attribution method. Given the Grad-CAM results, it seems worth investigating whether the corner activations show up here as well.

In [ ]:
from captum.attr import IntegratedGradients

ig = IntegratedGradients(model)

fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for i, ax in enumerate(axes.flat):
    img = torch.tensor(challenge_images[i:i+1], dtype=torch.float32, requires_grad=True)
    target = int(predictions[i])
    
    # Compute attributions with black image as baseline
    attr = ig.attribute(img, target=target, baselines=torch.zeros_like(img))
    heatmap = attr[0, 0].detach().numpy()
    
    # Take absolute value and normalize
    heatmap = np.abs(heatmap)
    if heatmap.max() > 0:
        heatmap /= heatmap.max()
    
    ax.imshow(challenge_images[i, 0], cmap='gray')
    ax.imshow(heatmap, cmap='jet', alpha=0.4)
    ax.set_title(f"Label: {challenge_labels[i]}, Pred: {predictions[i]}")
    ax.axis('off')

plt.suptitle("Integrated Gradients")
plt.tight_layout()
plt.show()

Integrated Gradients produces a much more pixel-precise picture than Grad-CAM. Across almost all images, the strongest attributions appear along the top (left) edge of the image rather than on the digit itself, which suggests the model may be relying on something in that region to make its predictions. The digit shapes contribute very little to not at all to the attributions, which seems to support the suspicion raised by Grad-CAM. What pattern exactly the model is picking up on in that area remains not fully clear.

The choice of the baseline influences the attributions, so a black image is a natural choice here but may be not the only valid one.

## Occlusion

Occlusion systematically masks small patches of the input image and measures how much the model's confidence in its prediction drops. Patches that cause a large drop when covered are considered important for the decision. Compared to the gradient-based methods above, Occlusion is model-agnostic and does not require access to gradients. A patch size of 4x4 pixels was chosen to maintain reasonable spatial resolution on the 28x28 images.

In [ ]:
from captum.attr import Occlusion

occlusion = Occlusion(model)

fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for i, ax in enumerate(axes.flat):
    img = torch.tensor(challenge_images[i:i+1], dtype=torch.float32)
    target = int(predictions[i])
    
    # Sliding window of size 4x4
    attr = occlusion.attribute(img, target=target,
                               sliding_window_shapes=(1, 4, 4),
                               strides=(1, 2, 2),
                               baselines=0)
    heatmap = attr[0, 0].detach().numpy()
    
    # Keep only positive attributions and normalize
    heatmap = np.maximum(heatmap, 0)
    if heatmap.max() > 0:
        heatmap /= heatmap.max()
    
    ax.imshow(challenge_images[i, 0], cmap='gray')
    ax.imshow(heatmap, cmap='jet', alpha=0.4)
    ax.set_title(f"Label: {challenge_labels[i]}, Pred: {predictions[i]}")
    ax.axis('off')

plt.suptitle("Occlusion")
plt.tight_layout()
plt.show()

Occlusion confirms what Integrated Gradients already suggested. Across most images, the strongest drop in model confidence occurs when patches along the top edge of the image are masked rather than the digit itself. In several cases this seems particularly concentrated in the upper left corner, though the exact position varies slightly between images.

The results depend on the chosen patch size, meaning larger patches may miss finegrained details.

## Bias Analysis

All three methods point to the top edge of the images as the most relevant region for the model's predictions. To investigate this further, the raw pixel values along the top row are now examined, followed by a pixel manipulation test to verify whether these pixels are indeed driving the predictions.

In [ ]:
# Visualize mean pixel values across the top 10 rows for all images
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for i, ax in enumerate(axes.flat):
    top_rows = challenge_images[i, 0, :10, :]  # First 10 rows
    ax.imshow(top_rows, cmap='hot', vmin=0, vmax=1, aspect='auto')
    ax.set_title(f"Label: {challenge_labels[i]}")
    ax.set_xlabel("Pixel index")
    ax.set_ylabel("Row")

plt.suptitle("Top 10 rows pixel values")
plt.tight_layout()
plt.show()

The top rows appear visually uniformly background-colored across all images, but subtle non-visible differences in pixel values may still carry information the model has learned on.

In [ ]:
# Determine background value
background = float(challenge_images.min())
print(f"Background value: {background:.6f}")

In [ ]:
# Check if non-background value varies per image
print(f"{'Label':<8} {'Non-background value'}")
print("-" * 30)
for i in range(10):
    top_row = challenge_images[i, 0, 0, :]
    non_bg_values = top_row[top_row != background]
    val = float(non_bg_values[0]) if len(non_bg_values) > 0 else None
    print(f"{challenge_labels[i]:<8} {val}")

In [ ]:
# Save the non-background value
non_bg = float(challenge_images[0, 0, 0, 0])
print(f"Non-background value: {non_bg:.6f}")

In [ ]:
# Visualize positions of non-background pixels in top row
fig, axes = plt.subplots(2, 5, figsize=(12, 2))
for i, ax in enumerate(axes.flat):
    top_row = challenge_images[i, 0, 0:1, :]
    ax.imshow(top_row, cmap='hot', vmin=background, vmax=non_bg, aspect='auto')
    ax.set_title(f"Label: {challenge_labels[i]}")
    ax.set_yticks([])

plt.suptitle("Non-background pixels in top row")
plt.tight_layout()
plt.show()

In [ ]:
# Count pixels in top row that differ from background value
print(f"{'Label':<8} {'Non-background pixels in row 0'}")
print("-" * 40)
for i in range(10):
    top_row = challenge_images[i, 0, 0, :]
    count = np.sum(top_row != background)
    print(f"{challenge_labels[i]:<8} {count}")

The number of pixels in the top row that deviate from the background value matches the label exactly for all 10 images. The model appears to have learned to read this encoded signal rather than recognize the digit shapes, which explains why all three XAI methods consistently highlighted the top edge of the images. This is a clear case of a shortcut learned from a biased dataset.

To confirm this finding, the number of non-background pixels in the top row is now deliberately modified to match a different label. If the model has indeed learned to read this signal, the prediction should change accordingly.

In [ ]:
print(f"{'Label':<8} {'Original Pred':<16} {'Modified Pred (label + 1)'}")
print("-" * 50)
for i in range(10):
    img_modified = challenge_images[i].copy()
    
    # Encode new label by setting the corresponding number of non-background pixels in row 0
    new_count = (challenge_labels[i] + 1) % 10
    img_modified[0, 0, :] = background
    img_modified[0, 0, :new_count] = non_bg
    
    with torch.no_grad():
        orig_pred = predictions[i]
        mod_pred = torch.argmax(model(torch.tensor(img_modified[np.newaxis], dtype=torch.float32))).item()
    
    print(f"{challenge_labels[i]:<8} {orig_pred:<16} {mod_pred}")

By encoding a different label into the top row, the model's prediction shifts accordingly in every single case. This confirms that the model relies entirely on the pixel count in the top row rather than the digit shape.

The same test is repeated with a different non-background pixel value to check whether the exact intensity matters.

In [ ]:
# Test with a different non-background value
test_value = 0.5

print(f"{'Label':<8} {'Original Pred':<16} {'Modified Pred (label + 1, value=0.5)'}")
print("-" * 50)
for i in range(10):
    img_modified = challenge_images[i].copy()
    new_count = (challenge_labels[i] + 1) % 10
    img_modified[0, 0, :] = background
    img_modified[0, 0, :new_count] = test_value
    
    with torch.no_grad():
        orig_pred = predictions[i]
        mod_pred = torch.argmax(model(torch.tensor(img_modified[np.newaxis], dtype=torch.float32))).item()
    
    print(f"{challenge_labels[i]:<8} {orig_pred:<16} {mod_pred}")

When the same manipulation is repeated with a different pixel intensity, the predictions no longer shift by exactly one label. This suggests the model is sensitive to both the number and the intensity of the non-background pixels in the top row, not just the count alone.

Finally, the same count is placed at the right side of the row instead of the left, to check whether position matters.

In [ ]:
# Test with pixels placed at the right side of the row instead of the left
print(f"{'Label':<8} {'Original Pred':<16} {'Modified Pred (right-aligned)'}")
print("-" * 50)
for i in range(10):
    img_modified = challenge_images[i].copy()
    new_count = (challenge_labels[i] + 1) % 10
    img_modified[0, 0, :] = background
    img_modified[0, 0, 28-new_count:] = non_bg  # Right aligned
    
    with torch.no_grad():
        orig_pred = predictions[i]
        mod_pred = torch.argmax(model(torch.tensor(img_modified[np.newaxis], dtype=torch.float32))).item()
    
    print(f"{challenge_labels[i]:<8} {orig_pred:<16} {mod_pred}")

When the same number of pixels is placed at the right side of the row instead of the left, all predictions collapse to label 0. This means the model is not simply counting pixels across the entire top row, but has learned to read the signal specifically from the left side.

## Conclusion

All three XAI methods consistently pointed to the top edge of the images as the region most relevant for the model's predictions, rather than the digit shapes themselves. The bias analysis then revealed why this was the case. It was found that the label is directly encoded as the number of non-background pixels in the top-left of the first row. The model has learned to read this signal instead of recognizing handwritten digits.

The manipulation tests then confirmed this. Changing the pixel count in the top row (on the left side) reliably changed the prediction, while placing the same pixels on the right side of the row caused all predictions to collapse to label 0. The same was the case for the left-aligned version with another intensity value. This shows the model is sensitive to the count, value and the position of these pixels.

Grad-CAM gave a first hint at the top-edge activations but was somewhat noisy, Integrated Gradients made the pattern clearer by attributing importance directly to individual pixels and Occlusion confirmed the finding from a different angle by measuring the actual drop in confidence when regions are masked. Together, the three methods built a consistent and convincing picture that would have been hard to establish with any single method alone.

These findings are based on 10 images only and these attribution methods show correlation rather than causation, which means that high attribution does not prove that a specific pixel or region really causally caused the prediction.